<a href="https://colab.research.google.com/github/rranagungorr/softito_egitim/blob/main/svm_analiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Support Vector Machine (SVM) - Hindistan Is Piyasasi Analizi

**Veri Seti:** `india_job_market_2024_2026.csv` (5000 is ilani, 17 sutun)

**Amaç:** Is ilanlarinin özelliklerine bakarak **Experience_Level** (tecrübe seviyesi) siniflandirmasi yapmak.

## SVM Teorisi

**Support Vector Machine**, siniflari birbirinden ayiran en iyi karar sinirini (hyperplane) bulan denetimli bir ögrenme algoritmasidir.

- **Hyperplane:** Siniflari ayiran düzlem/çizgi
- **Support Vector:** Hyperplane'e en yakin noktalar (siniri belirler)
- **Margin:** Hyperplane ile support vector'ler arasindaki mesafe (maximize edilir)
- **Kernel Trick:** Veriyi yüksek boyuta tasiyarak dogrusal olmayan sinirlar çizmeyi saglar

### Kernel Fonksiyonlari
| Kernel | Kullanim |
|--------|----------|
| **Linear** | Dogrusal ayrilabilen veriler |
| **RBF** | Dogrusal olmayan veriler (en popüler) |
| **Poly** | Polinom derecesi ile esneklik |

### Hiperparametreler
| Parametre | Anlami | Etkisi |
|-----------|--------|--------|
| **C** | Regularizasyon | Küçük C = genis margin (high bias), büyük C = dar margin (high variance) |
| **gamma** | RBF etki alani | Küçük = genis etki, büyük = dar etki |


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler,OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 1. Veri Setini Kesfetme ve Degiskenleri Tanima

Asagida veri setindeki tüm degiskenleri (sutunlari) ve özelliklerini inceleyecegiz.

In [ ]:
df = pd.read_csv('/content/india_job_market_2024_2026.csv')

print('=' * 70)
print('VERI SETI BOYUTU')
print('=' * 70)
print(f'Satir sayisi: {df.shape[0]}')
print(f'Sutun sayisi: {df.shape[1]}')
print()

print('=' * 70)
print('SUTUNLAR VE VERI TIPLERI')
print('=' * 70)
for col in df.columns:
    print(f"{'%-25s' % col} | dtype: {str(df[col].dtype):<15} | Benzersiz: {df[col].nunique()}")

VERI SETI BOYUTU
Satir sayisi: 5000
Sutun sayisi: 17

SUTUNLAR VE VERI TIPLERI
Job_ID                    | dtype: object          | Benzersiz: 5000
Job_Title                 | dtype: object          | Benzersiz: 30
Company                   | dtype: object          | Benzersiz: 53
Company_Type              | dtype: object          | Benzersiz: 4
Industry                  | dtype: object          | Benzersiz: 15
City                      | dtype: object          | Benzersiz: 17
Location_Tier             | dtype: object          | Benzersiz: 3
Experience_Level          | dtype: object          | Benzersiz: 5
Job_Type                  | dtype: object          | Benzersiz: 4
Work_Mode                 | dtype: object          | Benzersiz: 3
Salary_LPA                | dtype: float64         | Benzersiz: 728
Skills_Required           | dtype: object          | Benzersiz: 4817
Education_Required        | dtype: object          | Benzersiz: 8
Openings                  | dtype: int64           

In [ ]:
print('=' * 70)
print('DEGISKEN ISIMLERI VE ACILAMALARI')
print('=' * 70)
variable_info = {
    'Job_ID': 'Benzersiz is ilani numarasi',
    'Job_Title': 'Is unvani (Orn: Android Developer, QA Engineer)',
    'Company': 'Sirket adi',
    'Company_Type': 'Sirket turu (MNC, Startup, PSU/Govt, Indian Unicorn)',
    'Industry': 'Sektor (IT, FinTech, EdTech, HealthTech, vb.)',
    'City': 'Sehir',
    'Location_Tier': 'Sehir seviyesi (Tier 1, Tier 2, Remote)',
    'Experience_Level': 'Tecrube seviyesi (HEDEF DEGISKEN)',
    'Job_Type': 'Is turu (Full-Time, Part-Time, Internship, Contract)',
    'Work_Mode': 'Calisma sekli (Remote, On-Site, Hybrid)',
    'Salary_LPA': 'Yillik maas (LPA = Lakh Per Annum, 1 Lakh = 100,000 INR)',
    'Skills_Required': 'Gerekli beceriler',
    'Education_Required': 'Gerekli egitim (B.Tech, MCA, MBA, vb.)',
    'Openings': 'Acik pozisyon sayisi',
    'Applicants': 'Basvuran sayisi',
    'Company_Rating': 'Sirket puani (1.0 - 5.0)',
    'Date_Posted': 'Is ilani yayin tarihi'
}
for var, desc in variable_info.items():
    print(f"{'%-22s' % var} | {desc}")
print()

print('=' * 70)
print('HEDEF DEGISKEN: Experience_Level')
print('=' * 70)
print(df['Experience_Level'].value_counts().to_string())

DEGISKEN ISIMLERI VE ACILAMALARI
Job_ID                 | Benzersiz is ilani numarasi
Job_Title              | Is unvani (Orn: Android Developer, QA Engineer)
Company                | Sirket adi
Company_Type           | Sirket turu (MNC, Startup, PSU/Govt, Indian Unicorn)
Industry               | Sektor (IT, FinTech, EdTech, HealthTech, vb.)
City                   | Sehir
Location_Tier          | Sehir seviyesi (Tier 1, Tier 2, Remote)
Experience_Level       | Tecrube seviyesi (HEDEF DEGISKEN)
Job_Type               | Is turu (Full-Time, Part-Time, Internship, Contract)
Work_Mode              | Calisma sekli (Remote, On-Site, Hybrid)
Salary_LPA             | Yillik maas (LPA = Lakh Per Annum, 1 Lakh = 100,000 INR)
Skills_Required        | Gerekli beceriler
Education_Required     | Gerekli egitim (B.Tech, MCA, MBA, vb.)
Openings               | Acik pozisyon sayisi
Applicants             | Basvuran sayisi
Company_Rating         | Sirket puani (1.0 - 5.0)
Date_Posted            | Is ilan

In [ ]:
print('=' * 70)
print('KATEGORIK DEGISKENLERIN DAGILIMI')
print('=' * 70)
cat_columns = ['Company_Type', 'Industry', 'Location_Tier', 'Job_Type', 'Work_Mode', 'Education_Required']
for col in cat_columns:
    print(f'\\n{"-" * 50}')
    print(f'{col.upper()}')
    print(f'{"-" * 50}')
    for val, cnt in df[col].value_counts().items():
        print(f"  {'%-25s' % str(val)} : {cnt:5d} ({cnt/len(df)*100:5.2f}%)")

KATEGORIK DEGISKENLERIN DAGILIMI
\n--------------------------------------------------
COMPANY_TYPE
--------------------------------------------------
  MNC                       :  1717 (34.34%)
  Startup                   :  1430 (28.60%)
  Indian Unicorn            :  1210 (24.20%)
  PSU/Govt                  :   643 (12.86%)
\n--------------------------------------------------
INDUSTRY
--------------------------------------------------
  Information Technology    :  1545 (30.90%)
  FinTech                   :   543 (10.86%)
  E-Commerce                :   511 (10.22%)
  Banking & Finance         :   404 ( 8.08%)
  EdTech                    :   401 ( 8.02%)
  HealthTech                :   326 ( 6.52%)
  Consulting                :   261 ( 5.22%)
  Manufacturing             :   254 ( 5.08%)
  Government/PSU            :   202 ( 4.04%)
  Media & Entertainment     :   148 ( 2.96%)
  Gaming                    :   146 ( 2.92%)
  Logistics                 :   123 ( 2.46%)
  Retail         

In [ ]:
print('=' * 70)
print('SAYISAL DEGISKENLERIN ISTATISTIKLERI')
print('=' * 70)
num_cols = ['Salary_LPA', 'Openings', 'Applicants', 'Company_Rating']
print(df[num_cols].describe().to_string())

SAYISAL DEGISKENLERIN ISTATISTIKLERI
        Salary_LPA     Openings   Applicants  Company_Rating
count  5000.000000  5000.000000  5000.000000     5000.000000
mean     19.829440     3.642600   302.072000        3.698420
std      18.136741     4.046942   363.989613        0.424994
min       0.800000     1.000000    14.000000        2.500000
25%       6.800000     1.000000    99.000000        3.400000
50%      13.600000     2.000000   185.000000        3.800000
75%      25.600000     3.000000   321.000000        4.100000
max     115.400000    20.000000  2387.000000        4.300000


## 2. Veri Ön Isleme (Preprocessing)

SVM mesafe tabanli bir algoritma oldugu için:
1. **Kategorik degiskenler** sayisala çevrilmeli (LabelEncoder)
2. **Sayisal degiskenler** ölçeklendirilmeli (StandardScaler)
3. **Egitim/test** ayrimi yapilmali

In [ ]:
feature_cols = ['Company_Type', 'Industry', 'Location_Tier', 'Job_Type', 'Work_Mode',
                'Salary_LPA', 'Education_Required', 'Openings', 'Applicants', 'Company_Rating']
X = df[feature_cols].copy()
y = df['Experience_Level']

print('KULLANILACAK BAGIMSIZ DEGISKENLER (X):')
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")
print(f'\\nHEDEF DEGISKEN (y): Experience_Level')

KULLANILACAK BAGIMSIZ DEGISKENLER (X):
   1. Company_Type
   2. Industry
   3. Location_Tier
   4. Job_Type
   5. Work_Mode
   6. Salary_LPA
   7. Education_Required
   8. Openings
   9. Applicants
  10. Company_Rating
\nHEDEF DEGISKEN (y): Experience_Level


In [ ]:
"""
cat_cols = ['Company_Type', 'Industry', 'Location_Tier', 'Job_Type', 'Work_Mode', 'Education_Required']

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print('KATEGORIK DEGISKENLER SAYISALLASTIRILDI (LabelEncoder)')
print(f'\\n{"Degisken":<25} {"Siniflar":<50}')
print('\u2500' * 75)
for col in cat_cols:
    classes = label_encoders[col].classes_
    class_str = ', '.join([f"{i}:{c}" for i, c in enumerate(classes)])
    print(f"{col:<25} {class_str:<50}")

le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)
print(f'\\nHedef degisken (Experience_Level) kodlama:')
for i, cls in enumerate(le_y.classes_):
    print(f"  {i} -> {cls}")
"""

'\ncat_cols = [\'Company_Type\', \'Industry\', \'Location_Tier\', \'Job_Type\', \'Work_Mode\', \'Education_Required\']\n\nlabel_encoders = {}\nfor col in cat_cols:\n    le = LabelEncoder()\n    X[col] = le.fit_transform(X[col].astype(str))\n    label_encoders[col] = le\n\nprint(\'KATEGORIK DEGISKENLER SAYISALLASTIRILDI (LabelEncoder)\')\nprint(f\'\\n{"Degisken":<25} {"Siniflar":<50}\')\nprint(\'─\' * 75)\nfor col in cat_cols:\n    classes = label_encoders[col].classes_\n    class_str = \', \'.join([f"{i}:{c}" for i, c in enumerate(classes)])\n    print(f"{col:<25} {class_str:<50}")\n\nle_y = LabelEncoder()\ny_encoded = le_y.fit_transform(y)\nprint(f\'\\nHedef degisken (Experience_Level) kodlama:\')\nfor i, cls in enumerate(le_y.classes_):\n    print(f"  {i} -> {cls}")\n'

In [ ]:

from sklearn.preprocessing import OneHotEncoder
import pandas as pd

cat_cols = ['Company_Type', 'Industry', 'Location_Tier', 'Job_Type', 'Work_Mode', 'Education_Required']

# OneHotEncoder tanımla
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Kategorik sütunları dönüştür
ohe_array = ohe.fit_transform(X[cat_cols])

# Yeni sütun isimlerini oluştur
ohe_feature_names = ohe.get_feature_names_out(cat_cols)

# DataFrame'e çevir
ohe_df = pd.DataFrame(ohe_array, columns=ohe_feature_names, index=X.index)

# Orijinal kategorik sütunları düşür, OHE sütunlarını ekle
X = X.drop(columns=cat_cols)
X = pd.concat([X, ohe_df], axis=1)

print('KATEGORİK DEĞİŞKENLER SAYISALLAŞTIRILDI (OneHotEncoder)')
print(f'\n{"Değişken":<25} {"Oluşturulan Sütunlar":<60}')
print('─' * 85)
for col in cat_cols:
    # Bu değişkene ait sütunları filtrele
    cols_for_var = [f for f in ohe_feature_names if f.startswith(col + '_')]
    categories = [c.replace(col + '_', '') for c in cols_for_var]
    class_str = ', '.join([f"{i}:{c}" for i, c in enumerate(categories)])
    print(f"{col:<25} {class_str:<60}")


from sklearn.preprocessing import OrdinalEncoder

# Doğru sıralamayı manuel olarak tanımla
experience_order = [['Fresher (0-1 yr)', 'Junior (1-3 yrs)', 'Mid (3-6 yrs)', 'Senior (6-10 yrs)', 'Lead (10+ yrs)']]


oe_y = OrdinalEncoder(categories=experience_order, dtype=int)
y_encoded = oe_y.fit_transform(y.values.reshape(-1, 1)).ravel()

print(f'\nHedef değişken (Experience_Level) kodlama:')
for i, cls in enumerate(experience_order[0]):
    print(f"  {i} -> {cls}")


KATEGORİK DEĞİŞKENLER SAYISALLAŞTIRILDI (OneHotEncoder)

Değişken                  Oluşturulan Sütunlar                                        
─────────────────────────────────────────────────────────────────────────────────────
Company_Type              0:Indian Unicorn, 1:MNC, 2:PSU/Govt, 3:Startup              
Industry                  0:Automobile, 1:Banking & Finance, 2:Consulting, 3:E-Commerce, 4:EdTech, 5:FinTech, 6:Gaming, 7:Government/PSU, 8:HealthTech, 9:Information Technology, 10:Logistics, 11:Manufacturing, 12:Media & Entertainment, 13:Retail, 14:Telecom
Location_Tier             0:Remote, 1:Tier 1, 2:Tier 2                                
Job_Type                  0:Contract, 1:Full-Time, 2:Internship, 3:Part-Time          
Work_Mode                 0:Hybrid, 1:On-Site, 2:Remote                               
Education_Required        0:B.Com + Certification, 1:B.Sc (CS/IT), 2:B.Tech/B.E., 3:BCA, 4:M.Tech/M.E., 5:MBA, 6:MCA, 7:PhD

Hedef değişken (Experience_Level) kodla

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print('VERI AYRIMI (Train/Test)')
print(f'  Egitim seti  : {X_train.shape[0]} ornek (%80)')
print(f'  Test seti    : {X_test.shape[0]} ornek (%20)')
print(f'  Stratify     : Evet (sinif dagilimi korundu)')

VERI AYRIMI (Train/Test)
  Egitim seti  : 4000 ornek (%80)
  Test seti    : 1000 ornek (%20)
  Stratify     : Evet (sinif dagilimi korundu)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('VERI OLCEKLENDIRILDI (StandardScaler)')
print(f'  Ortalama (her sutun icin): 0')
print(f'  Standart sapma (her sutun icin): 1')
print(f'  X_train_scaled boyutu: {X_train_scaled.shape}')
print(f'  X_test_scaled boyutu : {X_test_scaled.shape}')

VERI OLCEKLENDIRILDI (StandardScaler)
  Ortalama (her sutun icin): 0
  Standart sapma (her sutun icin): 1
  X_train_scaled boyutu: (4000, 41)
  X_test_scaled boyutu : (1000, 41)


## 3. SVM Modelleri

3 farkli kernel ile SVM egitip karsilastiracagiz:
- **Linear SVM** (dogrusal)
- **RBF SVM** (radyal tabanli - varsayilan)
- **Polynomial SVM** (polinom)

Ardindan **GridSearchCV** ile en iyi parametreleri bulup modeli optimize edecegiz.

In [ ]:
print('=' * 50)
print('LINEAR SVM')
print('=' * 50)
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_linear):.4f}')
print(f'Destek vektor sayisi: {svm_linear.n_support_.sum()}')
print('\nSiniflandirma Raporu:')
print(classification_report(y_test, y_pred_linear, target_names=le_y.classes_))

LINEAR SVM
Test dogrulugu: 0.8310
Destek vektor sayisi: 1780

Siniflandirma Raporu:
                   precision    recall  f1-score   support

 Fresher (0-1 yr)       0.90      0.90      0.90       204
 Junior (1-3 yrs)       0.79      0.88      0.83       251
   Lead (10+ yrs)       0.87      0.80      0.84       276
    Mid (3-6 yrs)       0.78      0.77      0.77       166
Senior (6-10 yrs)       0.78      0.77      0.77       103

         accuracy                           0.83      1000
        macro avg       0.83      0.82      0.82      1000
     weighted avg       0.83      0.83      0.83      1000



In [ ]:
print('=' * 50)
print('RBF SVM (Varsayilan)')
print('=' * 50)
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_rbf):.4f}')
print(f'Destek vektor sayisi: {svm_rbf.n_support_.sum()}')
print('\nSiniflandirma Raporu:')
print(classification_report(y_test, y_pred_rbf, target_names=le_y.classes_))

RBF SVM (Varsayilan)
Test dogrulugu: 0.6030
Destek vektor sayisi: 3798

Siniflandirma Raporu:
                   precision    recall  f1-score   support

 Fresher (0-1 yr)       0.58      0.43      0.49       204
 Junior (1-3 yrs)       0.48      0.69      0.57       251
   Lead (10+ yrs)       0.67      0.66      0.67       276
    Mid (3-6 yrs)       0.70      0.60      0.65       166
Senior (6-10 yrs)       0.83      0.58      0.69       103

         accuracy                           0.60      1000
        macro avg       0.65      0.59      0.61      1000
     weighted avg       0.63      0.60      0.60      1000



In [ ]:
print('=' * 50)
print('POLINOMIAL SVM (degree=3)')
print('=' * 50)
svm_poly = SVC(kernel='poly', degree=3, random_state=42)
svm_poly.fit(X_train_scaled, y_train)
y_pred_poly = svm_poly.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_poly):.4f}')

POLINOMIAL SVM (degree=3)
Test dogrulugu: 0.4090


## 4. Hiperparametre Optimizasyonu (GridSearchCV)

En iyi `C` ve `gamma` degerlerini bulmak için **GridSearchCV** kullaniyoruz.

- **C**: [0.1, 1, 10, 100]
- **gamma**: ['scale', 'auto', 0.1, 0.01]
- **kernel**: ['rbf']
- **Cross-validation**: 5-fold

In [ ]:
param_grid = {
    'C': [100,300,500],
    'gamma': ['scale', 'auto', 0.005,0.01, 0.02],
    'kernel': ['rbf']
}

print('ARANAN PARAMETRELER:')
for k, v in param_grid.items():
    print(f'  {k}: {v}')
print(f'\nToplam kombinasyon: {len(param_grid["C"]) * len(param_grid["gamma"])}')
print(f'Toplam model (5-fold CV ile): {len(param_grid["C"]) * len(param_grid["gamma"]) * 5}')

ARANAN PARAMETRELER:
  C: [100, 300, 500]
  gamma: ['scale', 'auto', 0.005, 0.01, 0.02]
  kernel: ['rbf']

Toplam kombinasyon: 15
Toplam model (5-fold CV ile): 75


In [ ]:
grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

print(f'\n{"=" * 50}')
print('EN IYI PARAMETRELER')
print('=' * 50)
print(f'C          : {grid_search.best_params_["C"]}')
print(f'gamma      : {grid_search.best_params_["gamma"]}')
print(f'kernel     : {grid_search.best_params_["kernel"]}')
print(f'CV skoru   : {grid_search.best_score_:.4f}')

Fitting 5 folds for each of 15 candidates, totalling 75 fits

EN IYI PARAMETRELER
C          : 100
gamma      : 0.005
kernel     : rbf
CV skoru   : 0.7870


In [ ]:
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)

print('=' * 50)
print('EN IYI MODEL SONUCLARI')
print('=' * 50)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Cross-validation skoru: {grid_search.best_score_:.4f}')
print(f'\nSiniflandirma Raporu:')
print(classification_report(y_test, y_pred_best, target_names=le_y.classes_))

EN IYI MODEL SONUCLARI
Test dogrulugu: 0.7670
Cross-validation skoru: 0.7870

Siniflandirma Raporu:
                   precision    recall  f1-score   support

 Fresher (0-1 yr)       0.84      0.83      0.84       204
 Junior (1-3 yrs)       0.75      0.81      0.78       251
   Lead (10+ yrs)       0.78      0.76      0.77       276
    Mid (3-6 yrs)       0.72      0.66      0.69       166
Senior (6-10 yrs)       0.70      0.72      0.71       103

         accuracy                           0.77      1000
        macro avg       0.76      0.76      0.76      1000
     weighted avg       0.77      0.77      0.77      1000



In [ ]:
print('=' * 50)
print('KARISIKLIK MATRISI (Confusion Matrix)')
print('=' * 50)
cm = confusion_matrix(y_test, y_pred_best)
print('Satirlar: GERCEK deger | Sutunlar: TAHMIN edilen deger')
print()
header = ' ' * 22 + ' '.join(f'{c:>18}' for c in le_y.classes_)
print(header)
print('\u2500' * len(header))
for i, row in enumerate(cm):
    print(f"{le_y.classes_[i]:<22}" + ' '.join(f'{val:>18}' for val in row))
print('Yorum: Kosegen (diagonal) ne kadar yuksekse model o kadar basarili.')

KARISIKLIK MATRISI (Confusion Matrix)
Satirlar: GERCEK deger | Sutunlar: TAHMIN edilen deger

                        Fresher (0-1 yr)   Junior (1-3 yrs)     Lead (10+ yrs)      Mid (3-6 yrs)  Senior (6-10 yrs)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Fresher (0-1 yr)                     170                 25                  6                  2                  1
Junior (1-3 yrs)                      18                203                 23                  2                  5
Lead (10+ yrs)                        11                 37                210                 15                  3
Mid (3-6 yrs)                          3                  5                 25                110                 23
Senior (6-10 yrs)                      1                  1                  4                 23                 74
Yorum: Kosegen (diagonal) ne kadar yuksekse model o kadar basarili.


In [ ]:
print('=' * 50)
print('DESTEK VEKTOR ANALIZI')
print('=' * 50)
print(f'Toplam destek vektor sayisi  : {best_svm.n_support_.sum()}')
print(f'Toplam egitim ornegi         : {len(X_train_scaled)}')
print(f'Destek vektor orani          : {best_svm.n_support_.sum() / len(X_train_scaled) * 100:.1f}%')
print()
print('Her sinif icin destek vektor sayisi:')
for i, cls in enumerate(le_y.classes_):
    print(f'  {cls:<20} : {best_svm.n_support_[i]} vektor')

DESTEK VEKTOR ANALIZI
Toplam destek vektor sayisi  : 2108
Toplam egitim ornegi         : 4000
Destek vektor orani          : 52.7%

Her sinif icin destek vektor sayisi:
  Fresher (0-1 yr)     : 435 vektor
  Junior (1-3 yrs)     : 591 vektor
  Lead (10+ yrs)       : 506 vektor
  Mid (3-6 yrs)        : 382 vektor
  Senior (6-10 yrs)    : 194 vektor


In [ ]:
print('=' * 50)
print('ORNEK TAHMIN (Test Setinden 10 Kayit)')
print('=' * 50)
print(f'{"No":<5} {"Tahmin":<22} {"Gercek":<22} {"Dogru?":<8}')
print('\u2500' * 55)
for i in range(10):
    tahmin = le_y.inverse_transform([best_svm.predict(X_test_scaled[i].reshape(1, -1))[0]])[0]
    gercek = le_y.inverse_transform([y_test[i]])[0]
    dogru = '\u2713' if tahmin == gercek else '\u2717'
    print(f"{i+1:<5} {tahmin:<22} {gercek:<22} {dogru:<8}")

ORNEK TAHMIN (Test Setinden 10 Kayit)
No    Tahmin                 Gercek                 Dogru?  
───────────────────────────────────────────────────────
1     Fresher (0-1 yr)       Fresher (0-1 yr)       ✓       
2     Fresher (0-1 yr)       Fresher (0-1 yr)       ✓       
3     Lead (10+ yrs)         Lead (10+ yrs)         ✓       
4     Lead (10+ yrs)         Lead (10+ yrs)         ✓       
5     Senior (6-10 yrs)      Mid (3-6 yrs)          ✗       
6     Mid (3-6 yrs)          Mid (3-6 yrs)          ✓       
7     Fresher (0-1 yr)       Fresher (0-1 yr)       ✓       
8     Junior (1-3 yrs)       Junior (1-3 yrs)       ✓       
9     Fresher (0-1 yr)       Fresher (0-1 yr)       ✓       
10    Lead (10+ yrs)         Lead (10+ yrs)         ✓       


## 5. Sonuc ve Degerlendirme

| Model | Dogruluk | Aciklama |
|-------|----------|----------|
| Linear SVM | %69.4 | Dogrusal sinir, yeterince esnek degil |
| RBF SVM (varsayilan) | %74.0 | Daha iyi, dogrusal olmayan iliskileri yakalar |
| Polynomial SVM (3.derece) | %60.4 | Asiri uyum veya yetersiz |
| **Optimize RBF SVM** | **%76.9** | **C=100, gamma=0.01 ile en iyi sonuc** |

**En iyi model** `C=100, gamma=0.01, kernel='rbf'` parametreleriyle **%76.9** dogruluk elde etmistir.

### Dikkat Cikarimlar:
- **Fresher** sinifi en yüksek recall'a (%90) sahip -> yeni mezunlari bulmakta basarili
- **Lead** sinifi en yüksek precision'a (%90) sahip -> Lead tahmin ettiginde genelde dogru
- **Senior** sinifi en düsük recall'a (%65) sahip -> Senior'lari kaçirma orani yüksek
- Toplam 2664 destek vektörü (egitim verisinin %%67'si) -> sinir oldukça karmasik
